# Faster R-CNN — Car Detection

This notebook fine-tunes a **Faster R-CNN** detector (ResNet-50 + FPN backbone, pretrained on COCO) on a small car-detection dataset. The pipeline covers:

1. Loading the images and bounding-box annotations.
2. Building a custom PyTorch `Dataset` and dataloader.
3. Replacing the model's classifier head with a 2-class predictor.
4. Training with SGD + a learning-rate scheduler.
5. Evaluating, plotting precision/recall curves, and visualising predictions.

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import shutil
import random
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_curve


## 2. Optional dependency

`scipy` is used later to smooth the training curves.

In [ ]:
pip install scipy

## 3. Inspect the local dataset folder

In [ ]:
!ls data

## 4. Dataset paths and bounding-box CSV

Set local paths to the training/testing images and the bounding-box annotations CSV.

In [ ]:
train_data_path = "data/training_images"
test_data_path = "data/testing_images"
train_bboxes_csv_path = "data/train_solution_bounding_boxes.csv"

# Load bounding boxes data
bboxes = pd.read_csv(train_bboxes_csv_path)

print(f"Number of training images: {len(os.listdir(train_data_path))}, Number of bounding boxes: {bboxes.shape[0]}")

# Display first few rows
bboxes.head()


## 5. Custom PyTorch dataset

Wrap the CSV + images into a `torch.utils.data.Dataset` returning `(image, target)` pairs in the format Faster R-CNN expects (`boxes`, `labels`, `image_id`, `area`, `iscrowd`).

In [ ]:
# Custom dataset class for Faster R-CNN
class CarDetectionDataset(Dataset):
    def __init__(self, dataframe, img_dir, transforms=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transforms = transforms
        # Group bboxes by image to handle multiple cars in one image
        self.image_groups = self.df.groupby('image')

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, idx):
        img_name = list(self.image_groups.groups.keys())[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # Read image
        img = Image.open(img_path).convert("RGB")

        # Get all bboxes for this image
        boxes_df = self.df[self.df['image'] == img_name]
        boxes = []
        for _, row in boxes_df.iterrows():
            xmin, ymin, xmax, ymax = row[1:].values
            boxes.append([xmin, ymin, xmax, ymax])

        # Convert to tensor
        boxes = torch.as_tensor(boxes, dtype=torch.float32)

        # Create labels tensor (1 for car, as this is a single-class problem)
        labels = torch.ones((len(boxes),), dtype=torch.int64)

        # Create target dictionary
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels

        # Apply transforms if any
        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

# Function to display image with bounding box
def display_image_with_bbox(image_path, bboxes):
    try:
        image = cv2.imread(image_path)
        if image is None:
            raise Exception(f"Failed to read image: {image_path}")
        else:
            for bbox in bboxes:
                x1, y1, x2, y2 = map(int, bbox)
                # Draw rectangle: (start_point, end_point, color, thickness)
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Display image
            plt.imshow(image_rgb)
            plt.axis("off")
            plt.title("Image with detected Car (if present)")
    except Exception as e:
        print(f"Error displaying image {image_path}: {e}")


## 6. Visualise random samples

Quick sanity check that the boxes line up with the cars.

In [ ]:
# Display some random images with bounding boxes
plt.figure(figsize=(35, 10))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    idx = random.randint(0, bboxes.shape[0] - 1)
    img_path, bbox = bboxes.loc[idx, "image"], tuple(bboxes.iloc[idx, 1:].values)
    img_path = os.path.join(train_data_path, img_path)
    display_image_with_bbox(img_path, [bbox])


## 7. Transforms and dataloaders

Convert images to tensors and split the dataset 80 / 20 into train and validation loaders.

In [ ]:
# Define transforms for our dataset
def get_transform():
    return transforms.Compose([
        transforms.ToTensor(),
    ])

# Create dataset and dataloader
dataset = CarDetectionDataset(bboxes, train_data_path, get_transform())

# Split dataset into train and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Create data loaders
batch_size = 4
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))  # This is needed for detection models
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)


## 8. Build the Faster R-CNN model

Load the COCO-pretrained `fasterrcnn_resnet50_fpn` and swap its classifier head for one that predicts only **background** and **car** (`num_classes = 2`).

In [ ]:
# Function to get the Faster R-CNN model
def get_faster_rcnn_model(num_classes):
    # Load pre-trained model
    model = fasterrcnn_resnet50_fpn(pretrained=True)

    # Get the number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

# Get the model with 2 classes (background and car)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_faster_rcnn_model(num_classes=2)
model.to(device)

# Set up optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
from tqdm.notebook import tqdm


## 9. Training and evaluation helpers

`train_one_epoch` performs gradient accumulation; `evaluate` computes the validation loss.

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, accumulation_steps=4):
    model.train()
    total_loss = 0

    # Use tqdm for progress tracking
    progress_bar = tqdm(data_loader, desc="Training")

    optimizer.zero_grad()  # Zero gradients at the start

    for i, (images, targets) in enumerate(progress_bar):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Normalize the loss to account for accumulation
        losses = losses / accumulation_steps

        # Backward pass
        losses.backward()

        # Only step the optimizer after accumulation_steps
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        # Track full loss (not the normalized one) for reporting
        total_loss += losses.item() * accumulation_steps

        # Update progress bar with current loss
        progress_bar.set_postfix(loss=losses.item() * accumulation_steps)

    # Step the optimizer for any remaining gradients
    if (len(data_loader) % accumulation_steps) != 0:
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / len(data_loader)

def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0

    progress_bar = tqdm(data_loader, desc="Validating")

    with torch.no_grad():
        for images, targets in progress_bar:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Set to train mode temporarily to compute losses
            model.train()
            loss_dict = model(images, targets)
            model.eval()

            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()

            # FIX: Set postfix as a dictionary
            progress_bar.set_postfix({'loss': round(losses.item(), 4)})

    return total_loss / len(data_loader)


## 10. Train the model

Run the optimisation loop, store the train/validation losses for each epoch and update the LR scheduler.

In [ ]:
# Define metrics storage
metrics = {
    'train_loss': [],
    'val_loss': []
}

num_epochs = 15
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # Train for one epoch
    train_loss = train_one_epoch(model, optimizer, train_loader, device)
    metrics['train_loss'].append(train_loss)

    # Update the learning rate
    lr_scheduler.step()

    # Evaluate on the validation dataset
    val_loss = evaluate(model, val_loader, device)
    metrics['val_loss'].append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


## 11. Inspect the learning-rate schedule

In [ ]:
# This will show the current LR(s)
lrs = [group['lr'] for group in optimizer.param_groups]
plt.figure(figsize=(6,3))
plt.plot(lrs, marker='o')
plt.title("Learning Rate(s) (current values)")
plt.xlabel("Parameter group index")
plt.ylabel("LR")
plt.grid(True)
plt.show()


## 12. Plot training metrics

In [ ]:
# Plot training metrics
plt.figure(figsize=(10, 6))
plt.plot(metrics['train_loss'], label='Train Loss')
plt.plot(metrics['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Faster R-CNN Training Metrics')
plt.legend()
plt.grid()
plt.show()


## 13. Smoothed training curves

Apply a Gaussian filter to the loss curves to make trends easier to read.

In [ ]:
try:
    from scipy.ndimage import gaussian_filter1d
    train_smooth = gaussian_filter1d(metrics['train_loss'], sigma=1)
    val_smooth = gaussian_filter1d(metrics['val_loss'], sigma=1)

    plt.figure(figsize=(10,6))
    plt.plot(train_smooth, label="Smoothed Train Loss", linewidth=3)
    plt.plot(val_smooth, label="Smoothed Val Loss", linewidth=3)
    plt.legend()
    plt.title("Smoothed Loss Curves")
    plt.grid(True)
    plt.show()
except Exception as e:
    print("scipy not available or smoothing failed:", e)


## 14. Save the trained model

In [ ]:
# Path inside Google Drive
trained_model_dir = "Models"
trained_model_path = os.path.join(trained_model_dir, "car_detection_faster_rcnn.pt")

# Create folder if not exists
os.makedirs(trained_model_dir, exist_ok=True)

# Save model
torch.save(model.state_dict(), trained_model_path)

print(f"Model saved successfully at: {trained_model_path}")


## 15. Inference helpers

Run a forward pass on a single image and return the predicted boxes, labels and scores.

In [ ]:
# Function to perform inference and calculate metrics
def get_prediction(model, img_tensor, threshold=0.5):
    model.eval()
    with torch.no_grad():
        prediction = model([img_tensor.to(device)])

    # Get only predictions with score > threshold
    boxes = prediction[0]['boxes'][prediction[0]['scores'] > threshold].cpu().detach().numpy()
    scores = prediction[0]['scores'][prediction[0]['scores'] > threshold].cpu().detach().numpy()

    return boxes, scores

# Calculate mean Average Precision (simplified version)
def calculate_map(model, data_loader, iou_threshold=0.5, confidence_threshold=0.5):
    model.eval()
    all_precisions = []

    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)

            # Get predictions
            predictions = model(images)

            for i, prediction in enumerate(predictions):
                # Filter predictions by confidence threshold
                keep = prediction['scores'] > confidence_threshold
                boxes = prediction['boxes'][keep].cpu().numpy()
                scores = prediction['scores'][keep].cpu().numpy()

                # Get ground truth boxes
                gt_boxes = targets[i]['boxes'].cpu().numpy()

                # Calculate IoU for each pair of predicted and ground truth box
                if len(boxes) > 0 and len(gt_boxes) > 0:
                    # Count true positives and false positives
                    tp = 0
                    fp = 0

                    for box, score in zip(boxes, scores):
                        # Calculate IoU with all ground truth boxes
                        ious = []
                        for gt_box in gt_boxes:
                            # Calculate intersection
                            x1 = max(box[0], gt_box[0])
                            y1 = max(box[1], gt_box[1])
                            x2 = min(box[2], gt_box[2])
                            y2 = min(box[3], gt_box[3])

                            intersection = max(0, x2 - x1) * max(0, y2 - y1)

                            # Calculate areas
                            box_area = (box[2] - box[0]) * (box[3] - box[1])
                            gt_box_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])

                            # Calculate union
                            union = box_area + gt_box_area - intersection

                            # Calculate IoU
                            iou = intersection / union if union > 0 else 0
                            ious.append(iou)

                        # If any IoU is above threshold, it's a true positive
                        if len(ious) > 0 and max(ious) >= iou_threshold:
                            tp += 1
                        else:
                            fp += 1

                    # Calculate precision
                    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    all_precisions.append(precision)

    # Calculate mAP
    mAP = sum(all_precisions) / len(all_precisions) if all_precisions else 0
    return mAP

# Calculate metrics on validation set
mAP = calculate_map(model, val_loader)
print(f"\nmAP@0.5: {mAP:.4f}")


## 16. Precision-Recall analysis

In [ ]:
# Note: this is a simplified approach — for a robust PR curve you should build per-image TP/FP arrays.
def plot_pr_curve(model, data_loader):
    all_scores = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            preds = model(images)

            for pred, target in zip(preds, targets):
                scores = pred['scores'].cpu().numpy()
                # For simplified demonstration we set labels = 1 for each predicted box (not perfect)
                labels = np.ones_like(scores)
                all_scores.extend(scores)
                all_labels.extend(labels)

    if len(all_scores) == 0:
        print("No predictions collected for PR curve.")
        return

    precision, recall, _ = precision_recall_curve(all_labels, all_scores)

    plt.figure(figsize=(8,6))
    plt.plot(recall, precision)
    plt.title("Precision-Recall Curve (simplified)")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.grid(True)
    plt.show()

# Run:
plot_pr_curve(model, val_loader)


## 17. Visualise predictions on random test images

In [ ]:
# Function to visualize predictions on multiple random images
def visualize_multiple_random_images(model, test_data_path, num_images=4, threshold=0.5):
    # Get list of all image files
    image_files = [f for f in os.listdir(test_data_path) if f.endswith('.jpg')]

    # Create a subplot grid
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    # Select random images
    for i in range(min(num_images, len(image_files))):
        # Choose a random image
        idx = random.randint(0, len(image_files) - 1)
        image_name = image_files[idx]
        image_path = os.path.join(test_data_path, image_name)

        # Load and process the image
        img = Image.open(image_path).convert("RGB")
        img_tensor = get_transform()(img)

        # Get predictions
        boxes, scores = get_prediction(model, img_tensor, threshold)

        # Display image with bounding boxes
        image = cv2.imread(image_path)
        if image is not None:
            # Draw rectangles for all bounding boxes
            for j, (bbox, conf) in enumerate(zip(boxes, scores)):
                x1, y1, x2, y2 = map(int, bbox)
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)

                # Display confidence score
                conf_text = f"{conf:.2f}"
                cv2.putText(image, conf_text, (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            # Convert to RGB for matplotlib
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Display on appropriate subplot
            axes[i].imshow(image_rgb)
            axes[i].axis("off")
            axes[i].set_title(f"Image: {image_name}")

            # Print detection information
            print(f"Image {i+1}: {image_name}")
            if len(boxes) > 0:
                for j, (bbox, conf) in enumerate(zip(boxes, scores)):
                    print(f"  Detection {j+1} - Bounding Box: {bbox}, Confidence: {conf:.2f}")
            else:
                print("  No objects detected")
        else:
            print(f"Failed to read image: {image_path}")
            axes[i].text(0.5, 0.5, "Failed to load image",
                        horizontalalignment='center', verticalalignment='center')

    plt.tight_layout()
    plt.show()

# Test the model on random test images
visualize_multiple_random_images(model, test_data_path, num_images=4)


## 18. Side-by-side prediction comparison

In [ ]:
def visualize_prediction_comparison(model, test_data_path, threshold=0.5):
    image_files = [f for f in os.listdir(test_data_path) if f.endswith(".jpg")]
    if len(image_files) == 0:
        print("No images found in test_data_path.")
        return
    idx = random.randint(0, len(image_files) - 1)
    image_name = image_files[idx]
    image_path = os.path.join(test_data_path, image_name)

    img = Image.open(image_path).convert("RGB")
    img_tensor = get_transform()(img)

    boxes, scores = get_prediction(model, img_tensor, threshold)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Original
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Predicted (with colored boxes and scores)
    img_cv = cv2.imread(image_path)
    for bbox, conf in zip(boxes, scores):
        x1, y1, x2, y2 = map(int, bbox)
        cv2.rectangle(img_cv, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(img_cv, f"{conf:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,0,0), 2)

    img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    axes[1].imshow(img_rgb)
    axes[1].set_title(f"Prediction (threshold={threshold})")
    axes[1].axis("off")

    plt.show()

# Run it:
visualize_prediction_comparison(model, test_data_path)


## 19. Final results table

In [ ]:
results_df = pd.DataFrame({
    "Metric": ["Final Train Loss", "Final Val Loss", "mAP@0.5"],
    "Value": [metrics['train_loss'][-1] if len(metrics['train_loss'])>0 else None,
              metrics['val_loss'][-1] if len(metrics['val_loss'])>0 else None,
              mAP if 'mAP' in globals() else None]
})

print(results_df)
